In [ ]:
from huggingface_hub import login
token=''
login(token)

In [2]:
import torch
import torch.nn as nn
from transformers import AutoModel
import inspect

model = AutoModel.from_pretrained(
    "facebook/dinov3-vits16plus-pretrain-lvd1689m"
)

print("=== model loaded ===")
print(model)


Loading weights:   0%|          | 0/235 [00:00<?, ?it/s]

=== model loaded ===
DINOv3ViTModel(
  (embeddings): DINOv3ViTEmbeddings(
    (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
  )
  (rope_embeddings): DINOv3ViTRopePositionEmbedding()
  (layer): ModuleList(
    (0-11): 12 x DINOv3ViTLayer(
      (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (attention): DINOv3ViTAttention(
        (k_proj): Linear(in_features=384, out_features=384, bias=False)
        (v_proj): Linear(in_features=384, out_features=384, bias=True)
        (q_proj): Linear(in_features=384, out_features=384, bias=True)
        (o_proj): Linear(in_features=384, out_features=384, bias=True)
      )
      (layer_scale1): DINOv3ViTLayerScale()
      (drop_path): Identity()
      (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
      (mlp): DINOv3ViTGatedMLP(
        (gate_proj): Linear(in_features=384, out_features=1536, bias=True)
        (up_proj): Linear(in_features=384, out_features=1536, bias=True)
        (d

In [3]:
from src.make_gate import inject_gating, freeze_except_gate

In [4]:
model, hooks = inject_gating(model, gate_type="elementwise")

In [5]:
from src.model import dinosplus_classfier
vit=dinosplus_classfier(model,200)


In [6]:
print(vit)

dinosplus_classfier(
  (backbone): DINOv3ViTModel(
    (embeddings): DINOv3ViTEmbeddings(
      (patch_embeddings): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
    )
    (rope_embeddings): DINOv3ViTRopePositionEmbedding()
    (layer): ModuleList(
      (0-11): 12 x DINOv3ViTLayer(
        (norm1): LayerNorm((384,), eps=1e-05, elementwise_affine=True)
        (attention): DINOv3ViTAttention(
          (k_proj): Linear(in_features=384, out_features=384, bias=False)
          (v_proj): Linear(in_features=384, out_features=384, bias=True)
          (q_proj): Linear(in_features=384, out_features=384, bias=True)
          (o_proj): GatedOutputProjection(
            (original_o_proj): Linear(in_features=384, out_features=384, bias=True)
            (gate_proj): Linear(in_features=384, out_features=384, bias=True)
          )
        )
        (layer_scale1): DINOv3ViTLayerScale()
        (drop_path): Identity()
        (norm2): LayerNorm((384,), eps=1e-05, elementwise_affine=True)


In [7]:
for i in vit.parameters():
    i.requires_grad=True

In [8]:
from datasets import load_dataset
from src.data import setdata
from torch.utils.data import DataLoader
data=load_dataset('zh-plus/tiny-imagenet')
train=setdata(data['train'])
test=setdata(data['valid'])
train_loader=DataLoader(train,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)
test_loader=DataLoader(test,batch_size=128,pin_memory=True,num_workers=4,shuffle=True)

In [9]:
import torch.nn as nn
import torch.optim as optim
import torch
optimizer=optim.Adam(vit.parameters(),lr=0.0001)
criterion=nn.CrossEntropyLoss()
scaler=torch.amp.GradScaler()

In [10]:
import torch
print(torch.cuda.get_device_name(0))
device= 'cuda' if torch.cuda.is_available() else 'cpu'

NVIDIA GeForce RTX 5060 Ti


In [11]:
import torch

torch.backends.cuda.enable_flash_sdp(True)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(False)

In [12]:
from src.training import train 
train(vit,train_loader,test_loader,50,criterion,scaler,device,optimizer,'full fine tuning vit dino v3 splus gated','/home/hyuksu/deep-learning-study/outputs/best dino v3 splus gated.pth')

  0%|          | 0/50 [00:00<?, ?it/s]

Test Loss=0.8125, Test Accuracy=0.7960
Epoch 0 | Train Acc=0.6340 | Train Loss=1.8995
Test Loss=0.6885, Test Accuracy=0.8180
Epoch 1 | Train Acc=0.8627 | Train Loss=0.5230


KeyboardInterrupt: 